# Multi-Compartment Homogeneous Fleet Vehicle Routing Problem (MIP)

In [1]:
import DataFrames, Plots, SparseArrays
using JuMP, HiGHS, CPLEX, LinearAlgebra, DataFrames

In [2]:
function parameter_data()
    K =  15        # Number of Trucks
    P = 3         # Number of Products
    
    Vc = 4          # Number of Customer 
    d = rand(5:22, Vc+1, Vc+1)
    for i in 1:Vc+1
        d[i, i] =0
    end
    d[end, :] .= d[:, end]
    
    Qp = [100, 100, 100]        # Compartment Capacity
    Qmax = sum(Qp) * ones(1, K)    # Maximum Capacity of each truck
    q = rand(5:20,Vc,P)         # Demand of the customers

    L = 2500                    # Maximum length a route can be covered by a truck
    return Vc, K, P, d, Qmax, q, Qp, L
end
Vc, K, P, d, Qmax, q, Qp, L = parameter_data()

(4, 15, 3, [0 6 … 21 16; 12 0 … 22 7; … ; 11 18 … 0 22; 16 7 … 22 0], [300.0 300.0 … 300.0 300.0], [9 13 13; 13 5 16; 16 7 7; 15 9 5], [100, 100, 100], 2500)

In [3]:
model = Model(CPLEX.Optimizer)
set_silent(model)
@variable(model, x[1:Vc+1, 1:Vc+1, 1:K] >= 0, Bin)
@variable(model, z[1:Vc, 1:K, 1:P] >= 0, Bin)
@variable(model, y[1:Vc+1, 1:K], Bin)
@variable(model, u[1:Vc, 1:K, 1:P])
@objective(model, Min, sum(d[i, j] * x[i, j, k] for i in 1:(Vc+1) for j in 1:(Vc+1) for k in 1:K if i != j))
@constraint(model, [i in 1:Vc], sum(y[i, :]) == 1)
@constraint(model, sum(y[Vc+1, :]) <= K)
@constraint(model, [j in 1:Vc+1, k in 1:K], sum(x[i, j, k] for i in 1:(Vc+1) if i!=j) == y[j, k])
@constraint(model, [i in 1:Vc+1, k in 1:K], sum(x[i, j, k] for j in 1:(Vc+1) if j!=i) == y[i, k])
@constraint(model, [k in 1:K], sum(q[i, p] * y[i, k] for i in 1:Vc for p in P) <= Qmax[k])
@constraint(model, [j in 1:Vc, k in 1:K, p in 1:P], z[j, k, p] <= sum(x[i, j, k] for i in 1:Vc+1 if i!=j))
@constraint(model, [j in 1:Vc, p in 1:P], sum(z[j, :, p]) == 1)
@constraint(model, [k in 1:K, p in 1:P], sum(z[j, k, p] * q[j, p] for j in 1:Vc) <= Qp[p])
@constraint(model, [k in 1:K], sum(d[i, j] * x[i, j, k] for i in 1:(Vc+1) for j in 1:(Vc+1) if i != j) <= L)
@constraint(model, [i in 1:Vc, j in 1:Vc, j!=i, k in 1:K, p in 1:P], u[i, k, p] - u[j, k, p] + Qp[p] * x[i, j, k] <= Qp[p] - q[i, p])
@constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], q[i, p] - u[i, k, p] <= 0)
@constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], Qp[p] - u[i, k, p] >= 0)
set_time_limit_sec(model, 10.0)
optimize!(model)
xVals = value.(x)
for k = 1:size(xVals, 3), i = 1:size(xVals, 1), j = 1:size(xVals, 2)
    if xVals[i, j, k] > 0
        println("x($i, $j, $k): ", xVals[i, j, k])
    end
end
termination_status(model)

x(1, 5, 1): 1.0
x(2, 3, 1): 1.0
x(3, 4, 1): 1.0
x(4, 1, 1): 1.0
x(5, 2, 1): 1.0


TIME_LIMIT::TerminationStatusCode = 12

In [ ]:
using Plots

loc_x = rand(Vc+1)    # x-coordinates
loc_y = rand(Vc+1)    # y-coordinates

# Create a scatter plot
scatter(loc_x[1:end-1], loc_y[1:end-1], color=:yellow, marker=:star, aspect_ratio=1, legend=:topright, label="Customers")

# Add the depot point to the plot
scatter!([loc_x[end]], [loc_y[end]], color=:red, marker=:diamond, markersize=6, label="Depot")

display(plot!())

for i in 1:Vc+1, j in 1:Vc+1, k in 1:K
    if xVals[i, j, k] > 0.5
        plot!([loc_x[i], loc_x[j]], [loc_y[i], loc_y[j]], line=:path, color=:blue, alpha=0.5, label="")
    end
end

# Display the plot
display(plot!())  # Assuming "figure" is your plot


In [11]:
mutable struct Data
    d::Matrix{Float64}
    q::Matrix{Int64}
    Qp::Vector{Int64}
    Vc::Int64
    P::Int64
    L::Int64
    # initializer
    function Data(d::Matrix{Int64}, q::Matrix{Int64}, Qp::Vector{Int64}, P, L)
        # number of customers
        Vc = size(q,1)-1 
        new(d, q, Qp, Vc, P, L)
    end

end

In [12]:
A = [1	3	4	3	10
2	6	8	6	20
3	6	8	6	20
4	9	12	9	30
5	3	4	3	10
6	6	8	6	20
7	9	12	9	30
8	9	12	9	30
9	3	4	3	10
10	6	8	6	20
11	0	0	0	0
]
P = 3
q = A[:, 2:4] # demand
Vc = size(q, 1)-1 
G = zeros(Int,Vc)
for i in 1:Vc
    G[i] = sum(q[i, p] for p in 1:P)
end

d = rand(5:22, Vc+1, Vc+1)
    for i in 1:Vc+1
        d[i, i] =0
    end
d[end, :] .= d[:, end]
Qp = [40, 40, 40]       # Compartment Capacity 
L = 2500
data = Data(d, q, Qp, P, L)

Data([0.0 7.0 … 8.0 21.0; 11.0 0.0 … 5.0 12.0; … ; 19.0 21.0 … 0.0 18.0; 21.0 12.0 … 18.0 0.0], [3 4 3; 6 8 6; … ; 6 8 6; 0 0 0], [40, 40, 40], 10, 3, 2500)

In [13]:
mutable struct NaiveMIP
    model::Model
    K::Int64
    x::Array{VariableRef, 3}
    y::Matrix{VariableRef}
    z::Array{VariableRef, 3}
    u::Array{VariableRef, 3}

    # initializer
    function NaiveMIP(data::Data)
        
        d = data.d
        q = data.q
        Vc = data.Vc
        P = data.P
        L = data.L
        K = Vc
        Qp = data.Qp
        Qmax = sum(Qp) * ones(1, K)

        model = Model(CPLEX.Optimizer) # using HiGHS optimizer to avoid cplex 1217 error
        @variable(model, x[1:Vc+1, 1:Vc+1, 1:K] >= 0, Bin)
        @variable(model, z[1:Vc, 1:K, 1:P] >= 0, Bin)
        @variable(model, y[1:Vc+1, 1:K], Bin)
        @variable(model, u[1:Vc, 1:K, 1:P])

        # Statement 1: Add constraints to `model`
        
        @constraint(model, [i in 1:Vc], sum(y[i, :]) == 1)
        @constraint(model, sum(y[Vc+1, :]) <= K)
        @constraint(model, [j in 1:Vc+1, k in 1:K], sum(x[i, j, k] for i in 1:(Vc+1) if i!=j) == y[j, k])
        @constraint(model, [i in 1:Vc+1, k in 1:K], sum(x[i, j, k] for j in 1:(Vc+1) if j!=i) == y[i, k])
        @constraint(model, tr_cp[k in 1:K], sum(G[i] * y[i, k] for i in 1:Vc) <= Qmax[k])
        @constraint(model, [j in 1:Vc, k in 1:K, p in 1:P], z[j, k, p] <= sum(x[i, j, k] for i in 1:Vc+1 if i!=j))
        @constraint(model, [j in 1:Vc, p in 1:P], sum(z[j, :, p]) == 1)
        @constraint(model, [k in 1:K, p in 1:P], sum(z[j, k, p] * q[j, p] for j in 1:Vc) <= Qp[p])
        @constraint(model, [k in 1:K], sum(d[i, j] * x[i, j, k] for i in 1:(Vc+1) for j in 1:(Vc+1) if i != j) <= L)
        @constraint(model, [i in 1:Vc, j in 1:Vc, j!=i, k in 1:K, p in 1:P], u[i, k, p] - u[j, k, p] + Qp[p] * x[i, j, k] <= Qp[p] - q[i, p])
        @constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], q[i, p] - u[i, k, p] <= 0)
        @constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], Qp[p] - u[i, k, p] >= 0)
        
        # Statement 2: Add objective to `model`
        @objective(model, Min, sum(d[i, j] * x[i, j, k] for i in 1:(Vc+1) for j in 1:(Vc+1) for k in 1:K if i != j))
    
        new(model, K, x, y, z, u)
    end
end

In [15]:
nmip = NaiveMIP(data)
set_time_limit_sec(nmip.model, 400.0)
optimize!(nmip.model)
xVals = value.(nmip.x)
for k = 1:size(xVals, 3), i = 1:size(xVals, 1), j = 1:size(xVals, 2)
    if xVals[i, j, k] > 0
        println("x($i, $j, $k): ", xVals[i, j, k])
    end
end

nmip = NaiveMIP(A JuMP Model
Minimization problem with:
Variables: 1920
Objective function type: AffExpr
`AffExpr`-in-`MathOptInterface.EqualTo{Float64}`: 260 constraints
`AffExpr`-in-`MathOptInterface.GreaterThan{Float64}`: 300 constraints
`AffExpr`-in-`MathOptInterface.LessThan{Float64}`: 3651 constraints
`VariableRef`-in-`MathOptInterface.GreaterThan{Float64}`: 1510 constraints
`VariableRef`-in-`MathOptInterface.ZeroOne`: 1620 constraints
Model mode: AUTOMATIC
CachingOptimizer state: EMPTY_OPTIMIZER
Solver name: CPLEX


Names registered in the model: tr_cp, u, x, y, z, 10, [x[1,1,1] x[1,2,1] x[1,3,1] x[1,4,1] x[1,5,1] x[1,6,1] x[1,7,1] x[1,8,1] x[1,9,1] x[1,10,1] x[1,11,1]; x[2,1,1] x[2,2,1] x[2,3,1] x[2,4,1] x[2,5,1] x[2,6,1] x[2,7,1] x[2,8,1] x[2,9,1] x[2,10,1] x[2,11,1]; x[3,1,1] x[3,2,1] x[3,3,1] x[3,4,1] x[3,5,1] x[3,6,1] x[3,7,1] x[3,8,1] x[3,9,1] x[3,10,1] x[3,11,1]; x[4,1,1] x[4,2,1] x[4,3,1] x[4,4,1] x[4,5,1] x[4,6,1] x[4,7,1] x[4,8,1] x[4,9,1] x[4,10,1] x[4,11,1]; x[5,1,1] x[5,2,1] x[5,3,1] x[5,4,1] x[5,5,1] x[5,6,1] x[5,7,1] x[5,8,1] x[5,9,1] x[5,10,1] x[5,11,1]; x[6,1,1] x[6,2,1] x[6,3,1] x[6,4,1] x[6,5,1] x[6,6,1] x[6,7,1] x[6,8,1] x[6,9,1] x[6,10,1] x[6,11,1]; x[7,1,1] x[7,2,1] x[7,3,1] x[7,4,1] x[7,5,1] x[7,6,1] x[7,7,1] x[7,8,1] x[7,9,1] x[7,10,1] x[7,11,1]; x[8,1,1] x[8,2,1] x[8,3,1] x[8,4,1] x[8,5,1] x[8,6,1] x[8,7,1] x[8,8,1] x[8,9,1] x[8,10,1] x[8,11,1]; x[9,1,1] x[9,2,1] x[9,3,1] x[9,4,1] x[9,5,1] x[9,6,1] x[9,7,1] x[9,8,1] x[9,9,1] x[9,10,1] x[9,11,1]; x[10,1,1] x[10,2,1] x[10,3,

NaiveMIP(A JuMP Model
Minimization problem with:
Variables: 1920
Objective function type: AffExpr
`AffExpr`-in-`MathOptInterface.EqualTo{Float64}`: 260 constraints
`AffExpr`-in-`MathOptInterface.GreaterThan{Float64}`: 300 constraints
`AffExpr`-in-`MathOptInterface.LessThan{Float64}`: 3651 constraints
`VariableRef`-in-`MathOptInterface.GreaterThan{Float64}`: 1510 constraints
`VariableRef`-in-`MathOptInterface.ZeroOne`: 1620 constraints
Model mode: AUTOMATIC
CachingOptimizer state: EMPTY_OPTIMIZER
Solver name: CPLEX
Names registered in the model: tr_cp, u, x, y, z, 10, [x[1,1,1] x[1,2,1] … x[1,10,1] x[1,11,1]; x[2,1,1] x[2,2,1] … x[2,10,1] x[2,11,1]; … ; x[10,1,1] x[10,2,1] … x[10,10,1] x[10,11,1]; x[11,1,1] x[11,2,1] … x[11,10,1] x[11,11,1];;; x[1,1,2] x[1,2,2] … x[1,10,2] x[1,11,2]; x[2,1,2] x[2,2,2] … x[2,10,2] x[2,11,2]; … ; x[10,1,2] x[10,2,2] … x[10,10,2] x[10,11,2]; x[11,1,2] x[11,2,2] … x[11,10,2] x[11,11,2];;; x[1,1,3] x[1,2,3] … x[1,10,3] x[1,11,3]; x[2,1,3] x[2,2,3] … x[2,10,3

In [ ]:
mutable struct Master
    model::Model
    w::Vector{VariableRef} # variables
    demand_constr::Vector{<:ConstraintRef} # constraints
    # VI::ConstraintRef   # constraints
    routes::Vector{Vector{Float64}}
    # cost::Vector{Vector{Int64}}
    R::Int64 # number of routes
end

In [ ]:
function Master(data)
    routes = Vector{Vector{Float64}}(undef,0)
    # cost = Vector{Vector{Int64}}(undef,0)
    n = data.Vc
    d = data.d
    cd = d[:, end]
    K=n
            
    # statement 1: construct initial naive configurations 
    for i=1:n
        a = zeros(Int, n)
        a[i] = Int(floor(1 / ones(Int,n)[i]))
        push!(routes, a)
    end
    
    
    R = length(routes) # statement 2: Let P denote the number of initial routes
    
    c = zeros(Int, R)
    for r in 1:R
        c[r] = 2*cd[r]
        # push!(cost, c)
    end

    # define empty model
    model = Model(HiGHS.Optimizer)  # Using HiGHS optimizer instead of CPLEX to avoid CPLEX 1217 error
    
    # mute the output
    set_silent(model)

    # statement 3: add variable `w`, objective function, and constraint named 'demand_constr' to `model`
    
    @variable(model, w[1:R]>=0)
    @objective(model, Min, sum(c[r] * w[r] for r in 1:R))
    @constraint(model, demand_constr[i in 1:n],  routes[i]' * w >= 1)
    # @constraint(model, SRI, sum(floor((1/2)*sum(routes[i][r] for i in 3))*w[r] for r in 1:R) <= floor(length(3)/2))

    # @constraint(model, VI,  sum(routes[i]/2 for i in 1:3) * w <= Int(floor(3/2)))
    # @constraint(model, VI, sum(`w[r] .- sum(routes[i] for i in 1:3)` for r in 1:R) <= 1)
    
    return Master(model, w, demand_constr, routes, R)
end

In [ ]:
mmip = Master(data)
@show mmip.model
@show mmip.routes
@show mmip.w;

In [ ]:
optimize!(mmip.model)
solution_summary(mmip.model)

In [ ]:
mutable struct Sub
    model::Model
    a::Vector{VariableRef} # variables
    x::Matrix{VariableRef}
end

function Sub(data)

    d = data.d
    q = data.q
    P = data.P
    Vc = data.Vc
    G = zeros(Int,Vc)
    for i in 1:Vc
        G[i] = sum(q[i, p] for p in 1:P)
    end
    n=Vc
    L = data.L
    K = n
    Qp = data.Qp
    Qmax = sum(Qp)

    # initial dummy dual values
    π̂ = zeros(Float64, n)

    model = Model(HiGHS.Optimizer) # Using HiGHS optimizer instead of CPLEX to avoid CPLEX 1217 error

    set_silent(model)
    
    # statement 1: add constraints and objective to `model`
    
    @variable(model, x[1:n+1,1:n+1], Bin)
    @variable(model, y[1:n+1], Bin)
    @variable(model, z[1:n, 1:P], Bin)
    @variable(model, u[1:n, 1:P] >=0)
    @variable(model, a[1:n], Bin )

    # @constraint(model, [i in 1:Vc], sum(y[i]) == 1)
    # @constraint(model, sum(y[Vc+1]) <= K)
    @constraint(model, [j in 1:Vc+1], sum(x[i, j] for i in 1:(Vc+1) if i!=j) == y[j])
    @constraint(model, [i in 1:Vc+1], sum(x[i, j] for j in 1:(Vc+1) if j!=i) == y[i])
    @constraint(model, sum(G[i] * y[i] for i in 1:Vc) <= Qmax)
    @constraint(model, [j in 1:Vc, p in 1:P], z[j, p] <= sum(x[i, j] for i in 1:Vc+1 if i!=j))
    # @constraint(model, [j in 1:Vc, p in 1:P], z[j, p] == 1)
    @constraint(model, [p in 1:P], sum(z[j, p] * q[j, p] for j in 1:Vc) <= K * Qp[p])
    @constraint(model,  sum(d[i, j] * x[i, j] for i in 1:(Vc+1) for j in 1:(Vc+1) if i != j) <= L)
    @constraint(model, [i in 1:Vc, j in 1:Vc, j!=i, k in 1:K, p in 1:P], u[i, p] - u[j, p] + Qp[p] * x[i, j] <= Qp[p] - q[i, p])
    @constraint(model, [i in 1:Vc, p in 1:P], q[i, p] - u[i, p] <= 0)
    @constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], Qp[p] - u[i, p] >= 0)
    for i in 1:n
        @constraint(model, a[i] - sum(x[i, j] for j in 1:n+1) == 0)
    end
        
    @objective(model, Min, sum(d[i, j] * x[i, j] for i in 1:(Vc+1) for j in 1:(Vc+1))-sum(π̂[i] * a[i] for i in 1:n))
    
    
    return Sub(model, a, x)
end

In [ ]:
# generate the subproblem for CG
sub = Sub(data)

@show sub.model
@show sub.a;
@show sub.x;

In [ ]:
optimize!(sub.model)
solution_summary(sub.model)

In [ ]:
function runCG(master::Master, sub::Sub, data::Data)
    
    while true
        
        # statemet1: solve the restricted master problem
        
        # set_time_limit_sec(master.model, 60.0)
        optimize!(master.model)
        
        println(value.(master.w))
        println(objective_value(master.model))
        
        π = dual.(master.demand_constr) # statemet 2: get dual variables

        # statemet 3: update the subproblem objective function using the dual info
        set_objective_coefficient.(sub.model, sub.a, -π)
        
        # statemet 4: solve the subproblem
        # set_time_limit_sec(sub.model, 60.0)
        optimize!(sub.model)
        
        # statemet 5: obtain the reduced cost
        reduced_cost = objective_value(sub.model)
        if reduced_cost < -1e-8
            # statemet 6: add the column with negative reduced cost to the master model and routes
            â = value.(sub.a)
            xx = value.(sub.x)
            mc = sum(d[i, j] * xx[i,j] for i in 1:Vc+1, j in 1:Vc+1)
            push!(master.routes, â)
            push!(master.w, @variable(master.model, lower_bound = 0))
            set_objective_coefficient(master.model, master.w[end], mc)
            set_normalized_coefficient.(master.demand_constr, master.w[end], â)
            println("Found new route. Total routes = $(length(master.routes))")
        else
            break
            println("terminate")
        end
    end
end


In [ ]:
# generate the master problem for CG
master = Master(data)
sub = Sub(data)

tic = time()
runCG(master, sub, data)
toc = time()

solution_time = toc-tic
println("solution time: ", solution_time)

objective_value(master.model)

In [ ]:
master.routes

In [ ]:
println(value.(master.w))

In [ ]:
using Combinatorics
n =10
n_c =3
solve = Model(CPLEX.Optimizer)
@variable(solve, v[1:n], binary=true)
subsets = combinations(1:n, n_c)

for subset in subsets
    @constraint(solve, floor(sum(v[i] for i in subset) / 3) <= floor(3/2))
end

# Solve the model

In [ ]:
using JuMP
using HiGHS

# Create a JuMP model with HiGHS optimizer
model = Model(HiGHS.Optimizer)

# Define variables
@variable(model, x_12 >= 0)
@variable(model, x_13 >= 0)
@variable(model, x_23 >= 0)
@variable(model, x_24 >= 0)
@variable(model, x_34 >= 0)

# Add constraints
@constraint(model, n1, x_12 + x_13 == 14)
@constraint(model, n2, x_23 + x_24 == x_12)
@constraint(model, n3, x_34 - x_13- x_23 == -4)
@constraint(model, n4, x_24 + x_34 == 10)

# Define the coefficients for the objective function
coefficients = [2, 6, 1, 3, 5]

# Create an expression for the objective function
@expression(model, obj_expr, sum(coefficients[i] * x for (i, x) in enumerate([x_12, x_13, x_23, x_24, x_34])))

# Set the objective to minimize obj_expr
@objective(model, Min, obj_expr)

# Solve the primal model
optimize!(model)
println(value.(x_12))
println(value.(x_13))
println(value.(x_23))
println(value.(x_24))
println(value.(x_34))

In [ ]:
using JuMP
using CPLEX

# Create a JuMP model
model = Model(CPLEX.Optimizer)

# Define variables
@variable(model, x >= 0, Int)
@variable(model, y_plus >= 0)
@variable(model, y_minus >= 0)
@variable(model, z_plus >= 0)
@variable(model, z_minus >= 0)

# Set objective function
@objective(model, Max, 3*(z_plus - z_minus) - 5*(y_plus - y_minus) - 4*x)

# Add constraints
@constraint(model, x + y_plus - y_minus - z_plus + z_minus <= 5)
@constraint(model, -x - y_plus + y_minus + z_plus - z_minus <= -5)
@constraint(model, x + 2*(y_plus - y_minus) <= 10)
@constraint(model, z_plus - z_minus - x <= -2)

# Solve the optimization problem
optimize!(model)

# Get the optimal objective value
optimal_value = objective_value(model)

# Get the optimal variable values
optimal_x = value(x)
optimal_y_plus = value(y_plus)
optimal_y_minus = value(y_minus)
optimal_z_plus = value(z_plus)
optimal_z_minus = value(z_minus)

# Print results
println("Optimal Objective Value: ", optimal_value)
println("Optimal x: ", optimal_x)
println("Optimal y_plus: ", optimal_y_plus)
println("Optimal y_minus: ", optimal_y_minus)
println("Optimal z_plus: ", optimal_z_plus)
println("Optimal z_minus: ", optimal_z_minus)
